# Snakebite Species Identification: MVP Development Pipeline

This notebook establishes an end-to-end Machine Learning and data processing pipeline designed for regional snake species identification. It covers dataset ingestion, targeted geographical filtering, data pipeline remapping, transfer learning using EfficientNet, and an inference utility for downstream applications.

## Step 1: Ingesting Datasets from Kaggle

We utilize `kagglehub` to programmatically download two specialized datasets:
1. **Geographic Distribution Dataset**: Contains species-specific metadata alongside spatial distributions.
2. **Images Dataset**: Contains categorized images for 60+ snake species.

In [ ]:
import kagglehub

# Ingest geographic distribution metadata
geo_path = kagglehub.dataset_download("saidmassinissafazez/snakes-species-details-dataset")

# Ingest high-resolution species image repository
species_path = kagglehub.dataset_download("shouvikdey21/giant-snake-data60-species-kaggles-biggest")

print("Geographic metadata path:", geo_path)
print("Species image repository path:", species_path)

## Step 2: Mapping Regional Species Distributions

To adapt a global dataset for localized clinical utility, we parse the geographic metadata file (`t.json`). We extract and isolate species natively distributed within our target regional demo countries (e.g., Egypt, Nigeria, Ghana, Kenya, South Africa) and export a specialized JSON lookup table.

In [ ]:
import os
import json

# Locate the distribution JSON within the metadata directory
extracted_files = []
for root, _, files in os.walk(geo_path):
    for file in files:
        if file.endswith('.json'):
            extracted_files.append(os.path.join(root, file))

with open(extracted_files[0], 'r') as f:
    geo_data = json.load(f)

# Filter species based on localized operational vectors
target_countries = ['Egypt', 'Nigeria', 'Ghana', 'Kenya', 'South Africa']
african_mapping = {country: [] for country in target_countries}

for item in geo_data.get('species', []):
    countries = item.get('Geographic distribution', '')
    common_name = item.get('common name', item.get('Scientific name', ''))

    for country in target_countries:
        if country.lower() in str(countries).lower():
            african_mapping[country].append(common_name)

# Export lookup matrix for multi-agent system integration
with open('african_country_species_map.json', 'w') as f:
    json.dump(african_mapping, f, indent=4)

print("Successfully saved 'african_country_species_map.json'")

## Step 3: Image Data Preprocessing & Augmentation

Image data must be transformed to match the input specifications expected by neural networks pre-trained on ImageNet. 
- **Training Pipeline**: Employs spatial augmentations (`RandomResizedCrop`, `RandomHorizontalFlip`, `RandomRotation`) to artificially expand dataset variance, mitigating overfitting.
- **Validation Pipeline**: Employs deterministic operations (`Resize`, `CenterCrop`) to preserve feature consistency during evaluation.

In [ ]:
import os
from torchvision import datasets, transforms

image_dataset_dir = os.path.join(species_path, "Snakes_Dataset")

# Configure ImageNet-standardized tensor transform transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Parse raw image directory into a structured dataset mapping
full_dataset = datasets.ImageFolder(root=image_dataset_dir, transform=data_transforms['train'])
print(f"Total base categories identified: {len(full_dataset.classes)}")

## Step 4: Class Isolation & Target Dataset Remapping

Out of the 62 available classes, our immediate MVP isolates a core group of high-risk clinical targets (`bamboo_pit_viper`, `indian_saw_scaled_viper`, `russell_s_viper`, `spectacled_cobra`). 

Because `ImageFolder` assigns categorical indices based on alphabetical order across all 62 directories (e.g., index `20`, `43`), passing raw targets straight to a loss function with 4 outputs would trigger out-of-bounds runtime errors. We implement a custom PyTorch `Dataset` wrapper to isolate target frames and remap their indexes to a clean range ($[0, N-1]$).

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# Define indices derived from the parent ImageFolder structural scan
selected_indices = [20, 43, 56, 58]
filtered_class_names = ['bamboo_pit_viper', 'indian_saw_scaled_viper', 'russell_s_viper', 'spectacled_cobra']

# Extract every sample index matching our target category filters
sub_indices = [i for i, (_, label) in enumerate(full_dataset) if label in selected_indices]

class RemappedSnakeDataset(Dataset):
    def __init__(self, base_dataset, indices, target_classes):
        self.base_dataset = base_dataset
        self.indices = indices
        # Remap discrete labels (e.g., {20: 0, 43: 1, 56: 2, 58: 3}) to prevent classification head overflow
        self.label_mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(target_classes)}

    def __getitem__(self, idx):
        img, old_label = self.base_dataset[self.indices[idx]]
        return img, self.label_mapping[old_label]

    def __len__(self):
        return len(self.indices)

# Initialize isolated dataset split
mvp_dataset = RemappedSnakeDataset(full_dataset, sub_indices, selected_indices)

# Execute data split (80% Train, 20% Validation)
train_size = int(0.8 * len(mvp_dataset))
val_size = len(mvp_dataset) - train_size
train_dataset, val_dataset = random_split(mvp_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Dataset Partitioning Complete -> Train samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

## Step 5: Transfer Learning via EfficientNet Architecture

We leverage a pre-trained **EfficientNet-B0** backbone for feature extraction. Because generic image features (edges, textures) transfer well, we freeze the base convolutional parameters (`requires_grad = False`) and replace the terminal classification head with a newly initialized linear layer mapped to our 4 distinct targets. This isolates backpropagation strictly to the final layer, speeding up training on limited compute resources.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing training pipeline on device context: {device}")

# Instantiate backbone architecture with pre-trained ImageNet weights
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze feature extraction backbone parameters
for param in model.parameters():
    param.requires_grad = False

# Reconfigure linear classification head for our target output size (4 classes)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 4)
model = model.to(device)

# Configure loss optimization parameters
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

## Step 6: Execute Model Training

We train the updated model across 5 epochs. At each step, mini-batches are passed through the network, loss is calculated via cross-entropy, and gradients are updated through the Adam optimizer. The final fine-tuned model weights are saved to disk as a deployment-ready `.pth` file.

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = 100.0 * correct / total
    print(f"Epoch {epoch+1}/{epochs} -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

# Serialize model weights for downstream application deployment
torch.save(model.state_dict(), 'snake_species_mvp.pth')
print("Model state dict exported successfully as 'snake_species_mvp.pth'")

## Step 7: Inference Utility Pipeline

This utility function encapsulates the preprocessing and inference steps needed for live predictions. It takes an un-augmented image file path, applies standard validation transforms, runs a forward pass through the model in evaluation mode (`eval()`), applies a Softmax function to calculate class probabilities, and returns the classification target alongside its confidence score.

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms, models

def predict_uploaded_snake(image_path, model_path='snake_species_mvp.pth'):
    if not image_path or not os.path.exists(image_path):
        return {"status": "Failure", "prediction": "None", "confidence": 0.0}

    # Apply identical evaluation transformations
    eval_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Reconstruct empty model architecture shell
    infer_model = models.efficientnet_b0()
    infer_model.classifier[1] = torch.nn.Linear(infer_model.classifier[1].in_features, 4)
    
    # Load weights onto CPU context for maximum portability
    infer_model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
    infer_model.eval()

    # Preprocess image and add a batch dimension via unsqueeze
    img = Image.open(image_path).convert('RGB')
    tensor = eval_transform(img).unsqueeze(0)

    # Chronological target mapping lookup array
    classes = ['bamboo_pit_viper', 'indian_saw_scaled_viper', 'russell_s_viper', 'spectacled_cobra']

    # Disable gradient tracking for rapid, low-memory inference
    with torch.no_grad():
        out = infer_model(tensor)
        probs = F.softmax(out, dim=1)
        prob, class_idx = torch.max(probs, 1)

    return {
        "status": "Success",
        "prediction": classes[class_idx.item()],
        "confidence": round(prob.item(), 4)
    }